# Model Training for INX Future Inc Employee Performance Prediction

This notebook focuses on training machine learning models to predict employee
performance ratings. Multiple algorithms are evaluated, and the best-performing
model is selected based on performance metrics and interpretability.

## Import Required Libraries

Multiple algorithms are used to ensure robust model selection.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## Load Processed Dataset

In [2]:
df = pd.read_csv(r"C:\Users\madha\Desktop\Employee Performance Analysis\data\processed\employee_performance_processed.csv")

X = df.drop(columns=['PerformanceRating'])
y = df['PerformanceRating']

## Train–Test Split



In [3]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_bal, y_bal = sm.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.25, random_state=42, stratify=y_bal
)

C:\Users\madha\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\madha\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\madha\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\madha\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

## Logistic Regression

In [4]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.850609756097561
              precision    recall  f1-score   support

           2       0.88      0.92      0.90       218
           3       0.81      0.77      0.79       219
           4       0.86      0.86      0.86       219

    accuracy                           0.85       656
   macro avg       0.85      0.85      0.85       656
weighted avg       0.85      0.85      0.85       656



C:\Users\madha\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Random Forest Model (Primary Model)

In [5]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.9542682926829268
              precision    recall  f1-score   support

           2       0.94      0.99      0.96       218
           3       0.95      0.93      0.94       219
           4       0.97      0.95      0.96       219

    accuracy                           0.95       656
   macro avg       0.95      0.95      0.95       656
weighted avg       0.95      0.95      0.95       656



## Gradient Boosting

In [6]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.9298780487804879
              precision    recall  f1-score   support

           2       0.92      0.99      0.95       218
           3       0.92      0.90      0.91       219
           4       0.96      0.89      0.93       219

    accuracy                           0.93       656
   macro avg       0.93      0.93      0.93       656
weighted avg       0.93      0.93      0.93       656



## Feature Importance

In [7]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance.head(10)

,Feature,Importance
16,EmpLastSalaryHikePercent,0.224715
9,EmpEnvironmentSatisfaction,0.217330
23,YearsSinceLastPromotion,0.076072
27,PromotionGap,0.074795
22,ExperienceYearsInCurrentRole,0.034926
5,EmpJobRole,0.032153
26,ExperienceRatio,0.029287
10,EmpHourlyRate,0.026808
4,EmpDepartment,0.023391
20,EmpWorkLifeBalance,0.021514


## Model Selection Summary

Random Forest outperformed Logistic Regression in terms of accuracy and
class-wise performance. Additionally, Random Forest provides feature importance,
which is essential for business interpretability. Therefore, Random Forest was
selected as the final model for employee performance prediction.

## Save Trained Model

In [8]:
import joblib

joblib.dump(rf, "employee_performance_model.pkl")

['employee_performance_model.pkl']